In [1]:
from typing import List
import pickle
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
from cv2.datasets import none
from narwhals import Boolean


In [ ]:

def open_ghost_data(file_name: str):
	inputs = []
	expected_outputs = []
	with open(file_name, "r", encoding="utf-8") as file:
		for line in file:
			parsed_line = line.split()

			if len(parsed_line) >= 4:
				inputs.append([float(x) for x in parsed_line[0:3]])
				expected_outputs.append(int(parsed_line[3]) - 1)
	return np.array(inputs), np.array(expected_outputs)




In [ ]:

train_inputs, train_outputs = open_ghost_data("../data/colors_train.txt")
test_inputs, test_outputs = open_ghost_data("../data/colors_test.txt")

In [ ]:
import pandas
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1, as_frame=False)
x_data, y_data = mnist.data, mnist.target
x_data = x_data / 255

In [ ]:


x_train, x_test = x_data[:10000], x_data[10000:11000]
y_train, y_test = y_data[:10000], y_data[10000:11000]
print(f"x_train: {x_train.shape}, y_train: {y_train.shape}")
print(f"x_test: {x_test.shape}, y_test: {y_test.shape}")

In [2]:
class Module:
	def forward(self, x):
		raise NotImplementedError

	def backward(self, grad):
		raise NotImplementedError

	def adjust(self, eta):
		raise NotImplementedError


class LinearLayer(Module):
	def __init__(self, input_size: int, output_size: int, weights: None | np.ndarray = None,
	             biases: None | np.ndarray = None, min_weight:None|float = None ,max_weight:None|float = None, dropout_rate:float = 0.0):
		if weights is not None:
			if weights.shape != (output_size, input_size):
				raise ValueError("niezgodne wymiary macierzy wag")
			self.W = np.copy(weights)
		else:
			self.W = np.random.uniform(min_weight, max_weight, (output_size, input_size))

		if biases is not None:
			biases = np.atleast_2d(biases)
			if biases.shape != (1, output_size):
					raise ValueError("niezgodne wymiary macierzy biasow")
			self.b = np.copy(biases)
		else:
			self.b = None
		self.cache_X = None
		self.cache_grad = None
		self.is_training = False
		self.dropout_rate = dropout_rate
		self.dropout_mask = None
	def forward(self, x):
		self.cache_X = np.atleast_2d(x)
		output = x @ self.W.T.copy()

		if self.b is not None:
			return output + self.b

		if self.dropout_rate > 0 and self.is_training:
			keep_rate = 1-self.dropout_rate
			self.dropout_mask = np.random.binomial(1,keep_rate,output.shape) *  (1/ keep_rate)
			output = output * self.dropout_mask
			print(f"dropout_mask : {self.dropout_mask}")
		else:
			self.dropout_mask = None
		print(f"output: {output}")
		return output

	def backward(self, grad):
		if self.dropout_mask is not None:
			grad = grad * self.dropout_mask
		self.cache_grad = grad
		print(f"gradient: {grad}")
		print(f"Wagi: {self.W}")
		return grad @ self.W

	def adjust(self, eta):
		grad_W = self.cache_grad.T @ self.cache_X
		self.W -= eta * grad_W
		if self.b is not None:
			grad_b = np.sum(self.cache_grad, axis=0, keepdims=True)
			self.b -= eta * grad_b


class ActivationLayer(Module):

	@staticmethod
	def relu(x: np.ndarray) -> np.ndarray:
		return np.maximum(x, 0)

	@staticmethod
	def relu_derivative(x: np.ndarray) -> np.ndarray:
		return np.where(x > 0, 1.0, 0)

	ACTIVATION_FUNCTIONS = {
		'ReLU': (relu, relu_derivative)
	}

	def __init__(self, activation_function):
		self.activation_function = self.ACTIVATION_FUNCTIONS[activation_function][0]
		self.derivative_function = self.ACTIVATION_FUNCTIONS[activation_function][1]
		self.cache_S = None

	def forward(self, S):
		self.cache_S = S
		return self.activation_function(S)

	def backward(self, grad):
		return grad * self.derivative_function(self.cache_S)

	def adjust(self, eta):
		pass


class NeuralNetworkException(Exception):
	def __init__(self, msg):
		self.messege = msg

	def get_message(self):
		return self.messege


class NeuralNetwork:

	def __init__(self, initial_input_size):
		self.layers: list[Module] = list()
		self.initial_input_size = initial_input_size
		self.last_layer_input_size = initial_input_size

	def add_layer(self, layer: Module):
		self.layers.append(layer)

	def save_model(self, filename: str):
		params = {}
		for i, layer in enumerate(self.layers):
			if isinstance(layer, LinearLayer):
				params[f'W_{i}'] = layer.W
				if layer.b is not None:
					params[f'b_{i}'] = layer.b
		np.savez(filename, **params)

	def load_model(self, filename: str):
		data = np.load(filename if filename.endswith('.npz') else filename + '.npz')
		for i, layer in enumerate(self.layers):
			if isinstance(layer, LinearLayer):
				layer.W = data[f'W_{i}']
				if f'b_{i}' in data:
					layer.b = data[f'b_{i}']

	def predict(self, x: np.ndarray | None):
		output = x
		for layer in self.layers:
			output = layer.forward(output)

		return output

	def fit(self, x, y, learning_rate) -> float:
		prediction = self.predict(x)
		error = prediction - y
		loss = float(np.mean(np.square(error)))
		grad = (2.0 / error.size) * error
		for layer in reversed(self.layers):
			grad = layer.backward(grad)

		for layer in self.layers:
			layer.adjust(learning_rate)

		return loss

	def switch_training(self, state:bool):
		for layer in self.layers:
			if hasattr(layer, 'is_training'):
				layer.is_training = state



	def evaluate(self,x,y):
		data_size = y.shape[0]
		correct_predictions_count = 0
		false = []
		for i in range(data_size):
			predicted_num = np.argmax(self.predict(x[i]))
			expected_num = y[i]
			if int(predicted_num) == int(expected_num):
				correct_predictions_count += 1
			else:
				false.append(i)
		return ((correct_predictions_count / data_size), false)




In [12]:
test_network = NeuralNetwork(3)
test_network.add_layer(LinearLayer(3, 5, min_weight=0.2, max_weight=0.5, dropout_rate=0.0))
test_network.add_layer(LinearLayer(5, 5, min_weight=0.2, max_weight=0.5, dropout_rate=0.5))
test_network.add_layer(LinearLayer(5, 3, min_weight=0.2, max_weight=0.5, dropout_rate=0.0))


In [14]:

test_x = np.array([[1],[1],[1]])
test_y = np.array([1,1,1])
test_network.switch_training(True)
for _ in range(1000):
	test_network.fit(test_x.T, test_y, 0.1)
test_network.switch_training(False)
prediction = test_network.predict(test_x.T)
print(prediction)


output: [[0.67095842 0.61110413 0.66739465 0.55496756 0.57654994]]
dropout_mask : [[0. 0. 0. 2. 2.]]
output: [[0.         0.         0.         1.51786097 1.16045227]]
output: [[0.76015713 0.76015713 0.76015713]]
gradient: [[-0.15989525 -0.15989525 -0.15989525]]
Wagi: [[0.25255837 0.26678262 0.21608489 0.31679521 0.24068723]
 [0.25255837 0.26678262 0.21608489 0.31679521 0.24068723]
 [0.25255837 0.26678262 0.21608489 0.31679521 0.24068723]]
gradient: [[-0.         -0.         -0.         -0.3039243  -0.23090847]]
Wagi: [[0.14928767 0.27606243 0.25482173 0.14964023 0.1436727 ]
 [0.21117469 0.11930499 0.26009658 0.22321071 0.22903802]
 [0.22890933 0.0940462  0.19448978 0.16273493 0.1556304 ]
 [0.2714004  0.23824616 0.25076699 0.22565148 0.24048064]
 [0.23884313 0.27585114 0.12773183 0.13489451 0.15833657]]
gradient: [[-0.13763607 -0.13610516 -0.10570854 -0.09972925 -0.10964916]]
Wagi: [[0.21065969 0.1318123  0.32848643]
 [0.17788941 0.23709959 0.19611513]
 [0.10663507 0.26835214 0.2924074